Step 1: Install Dependency

In [44]:
%pip install -q kagglehub

from pathlib import Path

import cv2
import kagglehub
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Step 2: Download data from kaggle
I use downloaded zip on my PC, this black is not very important

In [45]:
path = kagglehub.dataset_download("gpreda/chinese-mnist")

def resolve_chinese_mnist_paths(dataset_root):
    root = Path(dataset_root)
    csv_matches = sorted(root.rglob("chinese_mnist.csv"))
    if not csv_matches:
        raise FileNotFoundError(f"No chinese_mnist.csv under {root}")
    csv_path = csv_matches[0]
    img_dir = csv_path.parent / "data" / "data"
    if not img_dir.is_dir():
        jpgs = list(root.rglob("input_*.jpg"))
        if not jpgs:
            raise FileNotFoundError("No image folder with input_*.jpg")
        img_dir = jpgs[0].parent
    return csv_path, img_dir


DATASET_ROOT = Path(path)
CSV_PATH, IMG_DIR = resolve_chinese_mnist_paths(DATASET_ROOT)
print(CSV_PATH)
print(IMG_DIR)

df_all = pd.read_csv(CSV_PATH)
label_col = "code"
df_all["path"] = df_all.apply(
    lambda r: IMG_DIR / f"input_{int(r.suite_id)}_{int(r.sample_id)}_{int(r.code)}.jpg",
    axis=1,
)

C:\Users\hrole\.cache\kagglehub\datasets\gpreda\chinese-mnist\versions\7\chinese_mnist.csv
C:\Users\hrole\.cache\kagglehub\datasets\gpreda\chinese-mnist\versions\7\data\data


Step 3: Get subset for train and test

In [46]:
train_pool_df, test_df = train_test_split(
    df_all,
    test_size=1000,
    stratify=df_all[label_col],
    random_state=42,
)
train_pool_df = train_pool_df.reset_index(drop=True)

placeholder = np.zeros((len(train_pool_df), 1))
y_pool = train_pool_df[label_col].to_numpy()

splitter_5k = StratifiedShuffleSplit(
    n_splits=1, train_size=5000, random_state=42
)
for train_idx, i in splitter_5k.split(placeholder, y_pool):
    train_5k_df = train_pool_df.iloc[train_idx].reset_index(drop=True)

splitter_10k = StratifiedShuffleSplit(
    n_splits=1, train_size=10000, random_state=42
)
for train_idx, i in splitter_10k.split(placeholder, y_pool):
    train_10k_df = train_pool_df.iloc[train_idx].reset_index(drop=True)

Step 4: Reshape

In [47]:
X_list, y_list = [], []
for i, row in test_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_test = np.asarray(X_list, dtype=np.float32)
y_test = np.asarray(y_list)

X_list, y_list = [], []
for i, row in train_5k_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_train_5k = np.asarray(X_list, dtype=np.float32)
y_train_5k = np.asarray(y_list)

X_list, y_list = [], []
for i, row in train_10k_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_train_10k = np.asarray(X_list, dtype=np.float32)
y_train_10k = np.asarray(y_list)

LABELS = np.sort(np.unique(y_test))

Step 5&6: init and train


In [48]:
knn_5k = KNeighborsClassifier(n_neighbors=3).fit(X_train_5k, y_train_5k)
dt_5k = DecisionTreeClassifier().fit(X_train_5k, y_train_5k)
sgd_5k = SGDClassifier(max_iter=250).fit(X_train_5k, y_train_5k)
print(5000, "done")

knn_10k = KNeighborsClassifier(n_neighbors=3).fit(X_train_10k, y_train_10k)
dt_10k = DecisionTreeClassifier().fit(X_train_10k, y_train_10k)
sgd_10k = SGDClassifier(max_iter=250).fit(X_train_10k, y_train_10k)
print(10000, "done")


5000 done
10000 done


Step 7&8: Predit and Evaluation

In [49]:
print("acc, prec, rec, f1")

print("Dataset", 5000)
pred = knn_5k.predict(X_test)
print("KNN", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))
pred = dt_5k.predict(X_test)
print("DT", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))
pred = sgd_5k.predict(X_test)
print("SGD", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))
print()

print("Dataset", 10000)
pred = knn_10k.predict(X_test)
print("KNN", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))
pred = dt_10k.predict(X_test)
print("DT", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))
pred = sgd_10k.predict(X_test)
print("SGD", accuracy_score(y_test, pred), precision_score(y_test, pred, average="macro", zero_division=0), recall_score(y_test, pred, average="macro", zero_division=0), f1_score(y_test, pred, average="macro", zero_division=0))
print(confusion_matrix(y_test, pred, labels=LABELS))


acc, prec, rec, f1
Dataset 5000
KNN 0.351 0.5630910688958937 0.350444745967134 0.3582437396455548
[[40  7  4  0  1  1  3  2  0  0  3  0  1  4  0]
 [ 0 65  0  2  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 35 23  8  0  0  0  0  0  0  0  0  1  0  0]
 [ 0 35 11 20  0  0  1  0  0  0  0  0  0  0  0]
 [ 0 35 11  3 14  0  2  0  0  0  1  0  0  1  0]
 [ 1 12 33 11  0  6  0  1  0  0  1  0  0  1  0]
 [ 0 50  0  2  0  0 15  0  0  0  0  0  0  0  0]
 [ 0 29  5  3  1  1  2 16  0  0 10  0  0  0  0]
 [ 0  6  0  1  0  0  2  0 57  0  1  0  0  0  0]
 [ 0 25  4  2  1  0 12  6  0  9  0  1  2  4  0]
 [ 0 25  1  0  0  0  2  0  0  0 34  0  5  0  0]
 [ 2 21  6  5  5  1  4  1  0  2  3 12  1  3  0]
 [ 0 27  2  2  0  0  0  0  0  1 17  0 18  0  0]
 [ 1 29  4  1  0  2  9  0  2  0  5  0  1 12  0]
 [ 1 27  8  7  1  0  1  2  1  3  2  1  1  2 10]]
DT 0.292 0.2865407649287742 0.2914819840192974 0.2871677972204242
[[25  1  1  1  2  4  6  5  0  4  1  3  9  2  2]
 [ 0 48  7  3  2  0  0  1  1  2  1  0  2  0  0]
 [ 0  8 18 11  2  0